In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RESULTS = Path('../results')
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(exist_ok=True)

# ── Load data ─────────────────────────────────────────────────────────────────
p1  = pd.read_csv(RESULTS / 'phase1_results.csv')
p2a = pd.read_csv(RESULTS / 'phase2_at_results.csv')
p2b = pd.read_csv(RESULTS / 'phase2_atkd_results.csv')
p3  = pd.read_csv(RESULTS / 'phase3_results.csv')

p1['defense'] = 'none'
p1['phase']   = 1
# Ensure consistent columns
for col in ['defense', 'phase']:
    if col not in p2a.columns: p2a[col] = col
    if col not in p2b.columns: p2b[col] = col

df = pd.concat([p1, p2a, p2b], ignore_index=True)

COMPRESSIONS = ['fp32', 'int8', 'int4']
DEFENSES     = ['none', 'at', 'at_kd']
ATTACKS      = ['fgsm', 'pgd', 'patch']

# ── Style ─────────────────────────────────────────────────────────────────────
matplotlib.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

C_NONE  = '#e74c3c'
C_AT    = '#2980b9'
C_ATKD  = '#27ae60'
DEFENSE_COLORS  = {'none': C_NONE,  'at': C_AT,    'at_kd': C_ATKD}
DEFENSE_LABELS  = {'none': 'No Defense', 'at': 'AT', 'at_kd': 'AT+KD'}
ATTACK_LABELS   = {'fgsm': 'FGSM', 'pgd': 'PGD', 'patch': 'Patch'}
COMP_LABELS     = {'fp32': 'FP32', 'int8': 'INT8', 'int4': 'INT4'}
COMP_X          = {'fp32': 0, 'int8': 1, 'int4': 2}

print('Data loaded.')
print(f'Phase 1 rows  : {len(p1)}')
print(f'Phase 2a rows : {len(p2a)}')
print(f'Phase 2b rows : {len(p2b)}')
print(f'Phase 3 rows  : {len(p3)}')

## Figure 1 — Baseline Vulnerability: ASR vs Compression (Phase 1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

markers = {'fgsm': 'o', 'pgd': 's', 'patch': '^'}
attack_colors = {'fgsm': '#e67e22', 'pgd': '#8e44ad', 'patch': '#16a085'}

for ax, attack in zip(axes, ATTACKS):
    sub = p1[p1['attack'] == attack].set_index('compression').reindex(COMPRESSIONS)
    x = [COMP_X[c] for c in COMPRESSIONS]
    ax.plot(x, sub['asr'].values * 100,
            marker=markers[attack], color=attack_colors[attack],
            linewidth=2.5, markersize=9, label=ATTACK_LABELS[attack])
    for xi, yi, c in zip(x, sub['asr'].values * 100, COMPRESSIONS):
        ax.annotate(f'{yi:.1f}%', (xi, yi), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=9.5, color=attack_colors[attack])
    ax.set_xticks(x)
    ax.set_xticklabels([COMP_LABELS[c] for c in COMPRESSIONS])
    ax.set_title(f'{ATTACK_LABELS[attack]} Attack', fontweight='bold')
    ax.set_xlabel('Compression Level')
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))
    if ax == axes[0]:
        ax.set_ylabel('Attack Success Rate (ASR)')
    ax.axhline(100, color='gray', linestyle='--', linewidth=1, alpha=0.5)

fig.suptitle(
    'Figure 1: Baseline Attack Success Rate Across Compression Levels (No Defense)',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(FIGURES / 'fig1_baseline_asr.pdf')
plt.savefig(FIGURES / 'fig1_baseline_asr.png')
plt.show()
print('Saved fig1_baseline_asr')

## Figure 2 — Defense Recovery: Robust Accuracy After AT and AT+KD

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

bar_w = 0.25
x = np.arange(len(COMPRESSIONS))

for ax, attack in zip(axes, ATTACKS):
    for i, defense in enumerate(DEFENSES):
        if defense == 'none':
            src = p1[p1['attack'] == attack]
        elif defense == 'at':
            src = p2a[p2a['attack'] == attack]
        else:
            src = p2b[p2b['attack'] == attack]
        src = src.set_index('compression').reindex(COMPRESSIONS)
        vals = src['robust_acc'].values * 100
        bars = ax.bar(x + (i - 1) * bar_w, vals, bar_w,
                      label=DEFENSE_LABELS[defense],
                      color=DEFENSE_COLORS[defense], alpha=0.88,
                      edgecolor='white', linewidth=0.5)
        for bar, val in zip(bars, vals):
            if val > 3:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                        f'{val:.0f}', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels([COMP_LABELS[c] for c in COMPRESSIONS])
    ax.set_title(f'{ATTACK_LABELS[attack]} Attack', fontweight='bold')
    ax.set_xlabel('Compression Level')
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))
    if ax == axes[0]:
        ax.set_ylabel('Robust Accuracy')

handles = [mpatches.Patch(color=DEFENSE_COLORS[d], label=DEFENSE_LABELS[d]) for d in DEFENSES]
fig.legend(handles=handles, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.07),
           frameon=False, fontsize=11)
fig.suptitle(
    'Figure 2: Robust Accuracy After Defense Across Compression Levels',
    fontsize=13, fontweight='bold', y=1.13
)
plt.tight_layout()
plt.savefig(FIGURES / 'fig2_defense_recovery.pdf')
plt.savefig(FIGURES / 'fig2_defense_recovery.png')
plt.show()
print('Saved fig2_defense_recovery')

## Figure 3 — AT vs AT+KD Gap Across Compression

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, attack in zip(axes, ATTACKS):
    at   = p2a[p2a['attack'] == attack].set_index('compression').reindex(COMPRESSIONS)
    atkd = p2b[p2b['attack'] == attack].set_index('compression').reindex(COMPRESSIONS)
    x = [COMP_X[c] for c in COMPRESSIONS]

    ax.plot(x, at['robust_acc'].values * 100,
            marker='s', color=C_AT, linewidth=2.5, markersize=9, label='AT')
    ax.plot(x, atkd['robust_acc'].values * 100,
            marker='o', color=C_ATKD, linewidth=2.5, markersize=9, label='AT+KD')

    # Shade the gap
    ax.fill_between(x,
                    at['robust_acc'].values * 100,
                    atkd['robust_acc'].values * 100,
                    alpha=0.12, color=C_ATKD, label='KD advantage')

    for xi, a, b in zip(x, at['robust_acc'].values * 100, atkd['robust_acc'].values * 100):
        gap = b - a
        if abs(gap) > 0.5:
            ax.annotate(f'Δ{gap:+.1f}%', (xi, max(a, b)),
                        textcoords='offset points', xytext=(6, 4),
                        fontsize=8.5, color='#555')

    ax.set_xticks(x)
    ax.set_xticklabels([COMP_LABELS[c] for c in COMPRESSIONS])
    ax.set_title(f'{ATTACK_LABELS[attack]} Attack', fontweight='bold')
    ax.set_xlabel('Compression Level')
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))
    if ax == axes[0]:
        ax.set_ylabel('Robust Accuracy')
        ax.legend(frameon=False)

fig.suptitle(
    'Figure 3: AT vs AT+KD Robust Accuracy — Knowledge Distillation Advantage',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(FIGURES / 'fig3_at_vs_atkd_gap.pdf')
plt.savefig(FIGURES / 'fig3_at_vs_atkd_gap.png')
plt.show()
print('Saved fig3_at_vs_atkd_gap')

## Figure 4 — Robustness Degradation: Clean vs Robust Accuracy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, attack in zip(axes, ATTACKS):
    for defense in DEFENSES:
        if defense == 'none':
            src = p1[p1['attack'] == attack]
        elif defense == 'at':
            src = p2a[p2a['attack'] == attack]
        else:
            src = p2b[p2b['attack'] == attack]
        src = src.set_index('compression').reindex(COMPRESSIONS)
        x = [COMP_X[c] for c in COMPRESSIONS]
        clean  = src['clean_acc'].values * 100
        robust = src['robust_acc'].values * 100
        ax.plot(x, clean,  linestyle='--', color=DEFENSE_COLORS[defense],
                linewidth=1.5, alpha=0.6)
        ax.plot(x, robust, linestyle='-',  color=DEFENSE_COLORS[defense],
                linewidth=2.5, marker='o', markersize=7,
                label=DEFENSE_LABELS[defense])
        ax.fill_between(x, clean, robust, alpha=0.07, color=DEFENSE_COLORS[defense])

    ax.set_xticks(x)
    ax.set_xticklabels([COMP_LABELS[c] for c in COMPRESSIONS])
    ax.set_title(f'{ATTACK_LABELS[attack]} Attack', fontweight='bold')
    ax.set_xlabel('Compression Level')
    ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))
    if ax == axes[0]:
        ax.set_ylabel('Accuracy')
        ax.legend(frameon=False, title='Defense')

# Legend for line styles
from matplotlib.lines import Line2D
style_legend = [
    Line2D([0], [0], color='gray', linestyle='--', linewidth=1.5, label='Clean Acc'),
    Line2D([0], [0], color='gray', linestyle='-',  linewidth=2.5, label='Robust Acc'),
]
fig.legend(handles=style_legend, loc='upper right', bbox_to_anchor=(1.0, 1.02),
           frameon=False, fontsize=10)

fig.suptitle(
    'Figure 4: Clean vs Robust Accuracy — Robustness Gap by Defense and Compression',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(FIGURES / 'fig4_clean_vs_robust.pdf')
plt.savefig(FIGURES / 'fig4_clean_vs_robust.png')
plt.show()
print('Saved fig4_clean_vs_robust')

## Figure 5 — ASR Heatmap: All Experiments

In [ ]:
# Build unified ASR matrix: rows = compression, cols = defense × attack
rows = []
for compression in COMPRESSIONS:
    row = {'compression': COMP_LABELS[compression]}
    for defense in DEFENSES:
        for attack in ATTACKS:
            if defense == 'none':
                src = p1
            elif defense == 'at':
                src = p2a
            else:
                src = p2b
            val = src[(src['compression'] == compression) & (src['attack'] == attack)]['asr']
            col = f'{DEFENSE_LABELS[defense]}\n{ATTACK_LABELS[attack]}'
            row[col] = float(val.values[0]) if len(val) else float('nan')
    rows.append(row)

hm_df = pd.DataFrame(rows).set_index('compression')

fig, ax = plt.subplots(figsize=(13, 3.8))
sns.heatmap(
    hm_df.astype(float),
    annot=True, fmt='.2f', cmap='RdYlGn_r',
    vmin=0, vmax=1,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'ASR (higher = more vulnerable)', 'shrink': 0.8},
    ax=ax
)

# Draw vertical lines separating defense groups
for x in [3, 6]:
    ax.axvline(x, color='black', linewidth=2)

# Defense group labels above
for i, defense in enumerate(DEFENSES):
    ax.text(i * 3 + 1.5, -0.6, DEFENSE_LABELS[defense],
            ha='center', va='center', fontsize=11, fontweight='bold',
            transform=ax.get_xaxis_transform())

ax.set_ylabel('Compression', fontweight='bold')
ax.set_xlabel('')
ax.set_title('Figure 5: Attack Success Rate — Full Experiment Matrix (Phase 1 & 2)',
             fontsize=13, fontweight='bold', pad=30)
plt.tight_layout()
plt.savefig(FIGURES / 'fig5_asr_heatmap.pdf')
plt.savefig(FIGURES / 'fig5_asr_heatmap.png')
plt.show()
print('Saved fig5_asr_heatmap')

## Figure 6 — Phase 3: Combined Attack vs All Defenses

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: ASR heatmap for Phase 3
p3_pivot = p3.pivot(index='compression', columns='defense', values='asr')
p3_pivot = p3_pivot.reindex(COMPRESSIONS)[DEFENSES]
p3_pivot.index = [COMP_LABELS[c] for c in COMPRESSIONS]
p3_pivot.columns = [DEFENSE_LABELS[d] for d in DEFENSES]

sns.heatmap(
    p3_pivot.astype(float),
    annot=True, fmt='.2f', cmap='RdYlGn_r',
    vmin=0, vmax=1,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'ASR', 'shrink': 0.8},
    ax=axes[0]
)
axes[0].set_title('Combined Attack ASR\n(FGSM → PGD → Patch)', fontweight='bold')
axes[0].set_ylabel('Compression', fontweight='bold')
axes[0].set_xlabel('Defense')

# Right: clean accuracy under combined attack
p3_clean = p3.pivot(index='compression', columns='defense', values='clean_acc')
p3_clean = p3_clean.reindex(COMPRESSIONS)[DEFENSES]
p3_clean.index = [COMP_LABELS[c] for c in COMPRESSIONS]
p3_clean.columns = [DEFENSE_LABELS[d] for d in DEFENSES]

x = np.arange(len(COMPRESSIONS))
bar_w = 0.25
for i, defense in enumerate(DEFENSES):
    vals = p3_clean[DEFENSE_LABELS[defense]].values * 100
    axes[1].bar(x + (i - 1) * bar_w, vals, bar_w,
                label=DEFENSE_LABELS[defense],
                color=DEFENSE_COLORS[defense], alpha=0.88,
                edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels([COMP_LABELS[c] for c in COMPRESSIONS])
axes[1].set_ylabel('Clean Accuracy')
axes[1].set_xlabel('Compression Level')
axes[1].set_ylim(0, 115)
axes[1].yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))
axes[1].set_title('Clean Accuracy Under Combined Attack\n(model preserves clean acc)', fontweight='bold')
axes[1].legend(frameon=False)

fig.suptitle(
    'Figure 6: Phase 3 — Combined Attack (FGSM→PGD→Patch) vs All Defenses',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(FIGURES / 'fig6_phase3_combined.pdf')
plt.savefig(FIGURES / 'fig6_phase3_combined.png')
plt.show()
print('Saved fig6_phase3_combined')

## Figure 7 — Compression Cost: Clean Accuracy Degradation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Use fgsm rows — clean_acc is same across attacks for same compression+defense
for defense in DEFENSES:
    if defense == 'none':
        src = p1[p1['attack'] == 'fgsm']
    elif defense == 'at':
        src = p2a[p2a['attack'] == 'fgsm']
    else:
        src = p2b[p2b['attack'] == 'fgsm']
    src = src.set_index('compression').reindex(COMPRESSIONS)
    x = [COMP_X[c] for c in COMPRESSIONS]
    ax.plot(x, src['clean_acc'].values * 100,
            marker='o', color=DEFENSE_COLORS[defense], linewidth=2.5,
            markersize=9, label=DEFENSE_LABELS[defense])
    for xi, yi in zip(x, src['clean_acc'].values * 100):
        ax.annotate(f'{yi:.1f}%', (xi, yi),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=9, color=DEFENSE_COLORS[defense])

ax.set_xticks([0, 1, 2])
ax.set_xticklabels([COMP_LABELS[c] for c in COMPRESSIONS])
ax.set_ylabel('Clean Accuracy')
ax.set_xlabel('Compression Level')
ax.set_ylim(70, 108)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))
ax.set_title('Figure 7: Clean Accuracy vs Compression Level by Defense',
             fontsize=13, fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIGURES / 'fig7_clean_acc_compression.pdf')
plt.savefig(FIGURES / 'fig7_clean_acc_compression.png')
plt.show()
print('Saved fig7_clean_acc_compression')

## Figure 8 — Robustness Gap: How Much Does Each Defense Recover?

In [ ]:
# Robustness recovery = (robust_acc_defended - robust_acc_none) for each compression × attack
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, attack in zip(axes, ATTACKS):
    baseline = p1[p1['attack'] == attack].set_index('compression').reindex(COMPRESSIONS)['robust_acc']
    at_rob   = p2a[p2a['attack'] == attack].set_index('compression').reindex(COMPRESSIONS)['robust_acc']
    atkd_rob = p2b[p2b['attack'] == attack].set_index('compression').reindex(COMPRESSIONS)['robust_acc']

    at_recovery   = (at_rob   - baseline) * 100
    atkd_recovery = (atkd_rob - baseline) * 100

    x = np.arange(len(COMPRESSIONS))
    ax.bar(x - 0.2, at_recovery.values,   0.35, label='AT',    color=C_AT,   alpha=0.88, edgecolor='white')
    ax.bar(x + 0.2, atkd_recovery.values, 0.35, label='AT+KD', color=C_ATKD, alpha=0.88, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8)

    for xi, va, vb in zip(x, at_recovery.values, atkd_recovery.values):
        ax.text(xi - 0.2, va + (1 if va >= 0 else -3), f'{va:+.1f}',
                ha='center', fontsize=8, color=C_AT)
        ax.text(xi + 0.2, vb + (1 if vb >= 0 else -3), f'{vb:+.1f}',
                ha='center', fontsize=8, color=C_ATKD)

    ax.set_xticks(x)
    ax.set_xticklabels([COMP_LABELS[c] for c in COMPRESSIONS])
    ax.set_title(f'{ATTACK_LABELS[attack]} Attack', fontweight='bold')
    ax.set_xlabel('Compression Level')
    if ax == axes[0]:
        ax.set_ylabel('Robust Accuracy Recovery (pp)')
        ax.legend(frameon=False)

fig.suptitle(
    'Figure 8: Robustness Recovery vs Baseline — Percentage Points Gained by Defense',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(FIGURES / 'fig8_robustness_recovery.pdf')
plt.savefig(FIGURES / 'fig8_robustness_recovery.png')
plt.show()
print('Saved fig8_robustness_recovery')

In [ ]:
import os
figs = sorted(FIGURES.glob('*.png'))
print(f'Total figures saved: {len(figs)}')
for f in figs:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f.name:<45} {size_kb:.0f} KB')